# 1D Fermi-Hubbard Dynamics with Majorana Propagation

In the following we will be studying real-time dynamics of interacting fermions in a classical Heisenberg-picture algorithm. This notebook is based on the research of D'Anna *et. al.*, such that we will try to reproduce their results and then apply the logic to our Kagome lattice, similar to what we did in 'kagomespd.ipynb'.[1]\
Majorana Propagation (MP) is the fermionic equivalent of Sparse Pauli Dynamics, with the twist that it operates in the **Majorana-string basis**, where fermionic operators are expanded as sums of Majorana operators. \
\
**A brief introduction into Majorana basis and MP:** \
For a detailed introduction into this topic, it would be appropriate to consider going through the referred to work or some other suitable literature.\
Instead of mapping fermionic operators to Pauli strings as for the Jordan-Wigner transformation, now we will be mapping those fermionic operators to products of Majorana operators. Majorana operators themselves are defined through fermionic operators as 
$$ c_j = a_j^{\dagger} + a_j, \quad \tilde{c_j} = i(a_j^{\dagger} - a_j)$$
where they obey the anticommutation relations 
$$\left\{c_i, c_j \right\} = \left\{\tilde{c_i}, \tilde{c_j} \right\} = 2\delta_{ij}, \qquad \left\{\tilde{c_i}, c_j \right\} = 0.$$
The self-adjoint $(c_i = \tilde{c_i})$ Majorana operators are then used to construct hermitian Majorana strings, represented by the binary vector $v = \left\{0,1\right\}^{2N}$ as 
$$\Gamma \equiv (i)^{v^T \omega_L v} \cdot c_1^{v_1}\tilde{c_1}^{v_2}  \cdots c_N^{v_{2N-1}}\tilde{c_N}^{v_{2N}},$$
while the phase factor $(i)^{v^T \omega_L v} = \left\{i,1\right\}$ recovers hermiticity of $ \Gamma = \Gamma^\dagger$, where $(\omega_L)_{ij} \equiv \delta_{i<j}$ is a $2N \times 2N$ lower triangular matrix. The matrix multiplication in the exponent is to be understood as mod 2.\
Another notable definition are unpaired Majorana strings. A mode $i$ is defined to be unpaired if it contains the operator $c_i$ without $\tilde{c_i}$ or vice versa. \
The number operator $n_i$ is defined as $$n_i = a_i^\dagger a_i = \frac{1}{2}(1+ic_i\tilde{c_i}).$$\
and the hopping operators as $$a_i^\dagger a_j + a_j^\dagger a_i = \frac{1}{2}(ic_i\tilde{c_j} - ic_i\tilde{c_j}), \qquad i < j.$$\
Majorna strings are closed under multiplication $$\mu(v)\,\mu(v') = \zeta\cdot \mu(v+v'), \qquad \zeta\in\{\pm1,\pm i\}$$ where $v + v^\prime$ is the bitwise addition mod 2.\
An important property is that these strings do satisfy the anticommuation relation $$\Gamma \Gamma^\prime = (-1)^{v^T \omega_L v^\prime} \Gamma^\prime \Gamma.$$



In [ ]:
try:
    from spd.SparsePauliDynamics import *
    from spd.BaseOperatorRepresentation import * 
except ImportError:
    print("\n" + "="*60)
    print("REQUIRED PACKAGE 'spd' NOT FOUND")
    print("="*60)
    print("This notebook relies on the 'spd' package.")
    print("Please install it before proceeding:\n")
    print("   pip install git+https://github.com/tbegusic/spd.git")
    print("="*60 + "\n")
    raise

import numpy as np
import matplotlib.pyplot as plt
from qiskit_nature.second_q.operators import FermionicOp

We consider a 1D chain of 10 spinful sites, where we will be simulating the dynamics of an inital Néel eigenstate (no hoppings, only singly occupied sites) at $U/t = +\infty$ evolving under the quenched Fermi-Hubbard Hamiltonian $$H = -t \sum_{\langle i,j\rangle,\sigma\in\{\uparrow,\downarrow\}} \left(c_{i,\sigma}^\dagger c_{j,\sigma} + c_{j,\sigma}^\dagger c_{i,\sigma}\right) + U \sum_i n_{i\uparrow} n_{i\downarrow}$$ with $U/t = 1$ in first order Trotter expansion. 

In [ ]:
L = 10                   
t = 1.0
U = 1.0
dt = 0.12                   
T = 6.0                     
nsteps = int(T / dt)         

threshold = 1e-5

nmodes = 2 * L

nlist = [(i, i+1) for i in range(L-1)]

In [ ]:
ham_hop = FermionicOp({}, nmodes)
ham_rep = FermionicOp({}, nmodes)


for i, j in nlist:
    up_i = 2 * i
    up_j = 2 * j
    ham_hop += FermionicOp({f'+_{up_i} -_{up_j}': -t}, num_spin_orbitals=nmodes)
    ham_hop += FermionicOp({f'+_{up_j} -_{up_i}': -t}, num_spin_orbitals=nmodes)
    down_i = 2 * i + 1
    down_j = 2 * j + 1
    ham_hop += FermionicOp({f'+_{down_i} -_{down_j}': -t}, num_spin_orbitals=nmodes)
    ham_hop += FermionicOp({f'+_{down_j} -_{down_i}': -t}, num_spin_orbitals=nmodes)

for site in range(L):
    up_idx = 2 * site
    down_idx = 2 * site + 1
    n_up = FermionicOp({f'+_{up_idx} -_{up_idx}': 1}, num_spin_orbitals=nmodes)
    n_down = FermionicOp({f'+_{down_idx} -_{down_idx}': 1}, num_spin_orbitals=nmodes)
    ham_rep += U * n_up @ n_down

In [ ]:
central = 5             
obs_up_idx = 2 * central
obs_ferm = FermionicOp({f'+_{obs_up_idx} -_{obs_up_idx}': 1}, num_spin_orbitals=nmodes)
obs_majorana = MajoranaRepresentation.from_fermionic_op(obs_ferm)

h_hop_majorana = MajoranaRepresentation.fermionic_to_sparse_pauli_op(ham_hop)
h_rep_majorana = MajoranaRepresentation.fermionic_to_sparse_pauli_op(ham_rep)

op = (dt * h_hop_majorana) + (dt * h_rep_majorana)

In [ ]:
state = np.zeros(nmodes, dtype=int)
for site in range(L):
    if site % 2 == 0:
        state[2*site+1] = 1 #even gets spin down
    else:
        state[2*site] = 1  #odd gets spin up

In [ ]:
S_vals = [2, 4, 6, 8]
results = {}
string_counts = {}
for S in S_vals:
    countstr = []
    def process(obs_rep):
        """Compute expectation value of the observable representation with the initial state."""

        exp_val = MajoranaRepresentation.exp_val_comp_basis_state(obs_rep, state)

        countstr.append(obs_rep.bits.shape[0])

        trunc = obs_rep.nonp_count <= S
        obs_rep.bits = obs_rep.bits[trunc]
        obs_rep.coeffs = obs_rep.coeffs[trunc]

        return exp_val
    
    sim = Simulation(obs_majorana.copy(), op, threshold=threshold)

    r = sim.run_dynamics(nsteps, process=process, process_every=1)

    results[str(S)] = np.array(r)
    string_counts[str(S)] = np.array(countstr)


## References 

[1] D’Anna, Matteo, Jannes Nys, and Juan Carrasquilla. “Majorana String Simulation of Nonequilibrium Dynamics in Two-Dimensional Lattice Fermion Systems.” arXiv:2511.02809. Preprint, arXiv, November 4, 2025. https://doi.org/10.48550/arXiv.2511.02809.
